E-Commerce Sales Data Pipeline

In [0]:
data = [
 (101,"Laptop",50000,"North"),
 (102,"Phone",30000,"South"),
 (103,None,25000,"North"),
 (104,"Tablet",None,"East"),
 (101,"Laptop",50000,"North"),
 (105,"Headphone",10000,"West")
]

bronze_df = spark.createDataFrame(
    data,
    schema = ["order_id", "product", "amount", "region"]
)
bronze_df.show()

+--------+---------+------+------+
|order_id|  product|amount|region|
+--------+---------+------+------+
|     101|   Laptop| 50000| North|
|     102|    Phone| 30000| South|
|     103|     NULL| 25000| North|
|     104|   Tablet|  NULL|  East|
|     101|   Laptop| 50000| North|
|     105|Headphone| 10000|  West|
+--------+---------+------+------+



In [0]:
bronze_df.write.format("delta") \
    .mode("overwrite") \
        .save("/Volumes/workspace/default/my_volume/delta/bronze_sales/sales")

In [0]:
bronze_df = spark.read.format("delta") \
    .load("/Volumes/workspace/default/my_volume/delta/bronze_sales/sales")


In [0]:
silver_df = bronze_df.dropDuplicates()

In [0]:
silver_df = silver_df.fillna({
    "product":"Unknown",
    "amount": 0
})

In [0]:
from pyspark.sql.functions import col

silver_df = silver_df.withColumn(
    "Tax",
    col("amount") * 0.18
)

In [0]:
from pyspark.sql.functions import when,col

silver_df = silver_df.withColumn(
    "category",
    when(col("amount")>= 40000,"Premium") 
    .otherwise("Standard")
)

In [0]:
from pyspark.sql.functions import sum 
silver_df.agg(sum("amount").alias("total_salary")).show()

+------------+
|total_salary|
+------------+
|      115000|
+------------+



In [0]:
silver_df.write.format("delta") \
    .mode("overwrite") \
        .save("/Volumes/workspace/default/my_volume/delta/silver_sales/sales")

In [0]:
gold_df = spark.read.format("delta") \
    .load("/Volumes/workspace/default/my_volume/delta/silver_sales/sales")

In [0]:
gold_df.groupBy("region") \
    .agg(sum("amount").alias("revenue")) \
        .show()

+------+-------+
|region|revenue|
+------+-------+
|  East|      0|
| North|  75000|
| South|  30000|
|  West|  10000|
+------+-------+



In [0]:
gold_df.groupBy("product") \
    .agg(sum("amount").alias("revenue")) \
        .show()

+---------+-------+
|  product|revenue|
+---------+-------+
|   Tablet|      0|
|   Laptop|  50000|
|  Unknown|  25000|
|    Phone|  30000|
|Headphone|  10000|
+---------+-------+



In [0]:
from pyspark.sql.functions import * 
from pyspark.sql.window import Window

window_spec = Window.partitionBy("region") \
    .orderBy(col("amount").desc())

rank_df = gold_df.withColumn(
    "rnk",
    dense_rank().over(window_spec)
)

rank_df.filter(
    col("rnk") == 1
).show()

+--------+---------+------+------+------+--------+---+
|order_id|  product|amount|region|   Tax|category|rnk|
+--------+---------+------+------+------+--------+---+
|     104|   Tablet|     0|  East|   0.0|Standard|  1|
|     101|   Laptop| 50000| North|9000.0| Premium|  1|
|     102|    Phone| 30000| South|5400.0|Standard|  1|
|     105|Headphone| 10000|  West|1800.0|Standard|  1|
+--------+---------+------+------+------+--------+---+



In [0]:
gold_df.write.format("delta") \
    .mode("overwrite") \
        .save("/Volumes/workspace/default/my_volume/delta/gold_sales/sales")

In [0]:
bronze_df.show()

+--------+---------+------+------+
|order_id|  product|amount|region|
+--------+---------+------+------+
|     101|   Laptop| 50000| North|
|     102|    Phone| 30000| South|
|     103|     NULL| 25000| North|
|     104|   Tablet|  NULL|  East|
|     101|   Laptop| 50000| North|
|     105|Headphone| 10000|  West|
+--------+---------+------+------+



In [0]:
silver_df.show()

+--------+---------+------+------+------+--------+
|order_id|  product|amount|region|   Tax|category|
+--------+---------+------+------+------+--------+
|     101|   Laptop| 50000| North|9000.0| Premium|
|     102|    Phone| 30000| South|5400.0|Standard|
|     103|  Unknown| 25000| North|4500.0|Standard|
|     104|   Tablet|     0|  East|   0.0|Standard|
|     105|Headphone| 10000|  West|1800.0|Standard|
+--------+---------+------+------+------+--------+



In [0]:
gold_df.show()

+--------+---------+------+------+------+--------+
|order_id|  product|amount|region|   Tax|category|
+--------+---------+------+------+------+--------+
|     104|   Tablet|     0|  East|   0.0|Standard|
|     101|   Laptop| 50000| North|9000.0| Premium|
|     103|  Unknown| 25000| North|4500.0|Standard|
|     102|    Phone| 30000| South|5400.0|Standard|
|     105|Headphone| 10000|  West|1800.0|Standard|
+--------+---------+------+------+------+--------+



Customer Analytics Pipeline

In [0]:
data = [
 (1,"John",5000),
 (2,"Jane",12000),
 (3,"Mike",8000),
 (4,"Sara",25000),
 (5,"Tom",None),
 (2,"Jane",12000)
]

columns = [
 "customer_id",
 "customer_name",
 "purchase_amount"
]

bronze_df = spark.createDataFrame(
    data,
    columns
)

bronze_df.write.format("delta") \
    .mode("overwrite") \
        .save("/Volumes/workspace/default/my_volume/delta/bronze_customers/customers")

In [0]:
silver_df = spark.read.format("delta") \
    .load("/Volumes/workspace/default/my_volume/delta/bronze_customers/customers")


In [0]:
silver_df = silver_df.dropDuplicates()

In [0]:
silver_df = silver_df.fillna({
    "purchase_amount":0
})

In [0]:
from pyspark.sql.functions import when,col 
silver_df = silver_df.withColumn(
    "segment",
    when(col("purchase_amount") >=20000,"VIP") 
    .when(col("purchase_amount") >=10000,"Premium")
    .otherwise("Regular")
)

In [0]:
silver_df.write.format("delta") \
    .mode("overwrite") \
        .save("/Volumes/workspace/default/my_volume/delta/silver_customers/customers")  
display(silver_df)

customer_id,customer_name,purchase_amount,segment
1,John,5000,Regular
2,Jane,12000,Premium
3,Mike,8000,Regular
4,Sara,25000,VIP
5,Tom,0,Regular


In [0]:
from pyspark.sql.functions import avg

gold_avg_df = silver_df.agg(avg("purchase_amount").alias("avg_spend"))

In [0]:
gold_segment_df = silver_df.groupBy("segment").count()

In [0]:
from pyspark.sql.functions import sum 

gold_revenue_df = silver_df.groupBy("segment") \
    .agg(sum("purchase_amount").alias("revenue"))
    

In [0]:
from pyspark.sql.window import Window 
from pyspark.sql.functions import dense_rank

window_spec = Window.orderBy(
    col("purchase_amount").desc()
)

rank_df = silver_df.withColumn(
    "customer_rank",
    dense_rank().over(window_spec)
)

rank_df.show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+-----------+-------------+---------------+-------+-------------+
|customer_id|customer_name|purchase_amount|segment|customer_rank|
+-----------+-------------+---------------+-------+-------------+
|          4|         Sara|          25000|    VIP|            1|
|          2|         Jane|          12000|Premium|            2|
|          3|         Mike|           8000|Regular|            3|
|          1|         John|           5000|Regular|            4|
|          5|          Tom|              0|Regular|            5|
+-----------+-------------+---------------+-------+-------------+



In [0]:
rank_df.filter(
    col("customer_rank") ==1
).show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+-----------+-------------+---------------+-------+-------------+
|customer_id|customer_name|purchase_amount|segment|customer_rank|
+-----------+-------------+---------------+-------+-------------+
|          4|         Sara|          25000|    VIP|            1|
+-----------+-------------+---------------+-------+-------------+



In [0]:
gold_df = rank_df 

gold_df.write.format("delta") \
    .mode("overwrite") \
        .save("/Volumes/workspace/default/my_volume/delta/gold_customers/customers")
display(gold_df)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


customer_id,customer_name,purchase_amount,segment,customer_rank
4,Sara,25000,VIP,1
2,Jane,12000,Premium,2
3,Mike,8000,Regular,3
1,John,5000,Regular,4
5,Tom,0,Regular,5


In [0]:
display(silver_df)

customer_id,customer_name,purchase_amount,segment
1,John,5000,Regular
2,Jane,12000,Premium
3,Mike,8000,Regular
4,Sara,25000,VIP
5,Tom,0,Regular


In [0]:
display(bronze_df)

customer_id,customer_name,purchase_amount
1,John,5000
2,Jane,12000
3,Mike,8000
4,Sara,25000
5,Tom,null
2,Jane,12000


Banking Transaction Analytics Pipeline

In [0]:
data = [
 (101,"John",5000),
 (102,"Jane",25000),
 (103,"Mike",120000),
 (104,"Sara",8000),
 (105,"Tom",70000),
 (103,"Mike",120000),
 (106,"Alex",None)
]

columns = [
 "transaction_id",
 "customer_name",
 "amount"
]

bronze_df = spark.createDataFrame(
    data,
    columns
)

bronze_df.write.format("delta") \
    .mode("overwrite") \
        .save("/Volumes/workspace/default/my_volume/delta/bronze_Banking/bank_transactions")
display(bronze_df)

transaction_id,customer_name,amount
101,John,5000
102,Jane,25000
103,Mike,120000
104,Sara,8000
105,Tom,70000
103,Mike,120000
106,Alex,null


In [0]:
silver_df = spark.read.format("delta") \
    .load("/Volumes/workspace/default/my_volume/delta/bronze_Banking/bank_transactions")

In [0]:
silver_df = silver_df.dropDuplicates()

In [0]:
silver_df = silver_df.fillna({
    "amount": 0
})

In [0]:
from pyspark.sql.functions import when,col 

silver_df = silver_df.withColumn(
    "risk_categories",
    when(col("amount") >= 100000,"High Risk")
    .when(col("amount") >= 50000,"Medium Risk")
    .otherwise("Low Risk")
)

In [0]:
silve_df = silver_df.withColumn(
    "transcation_fee",
    col("amount") * 0.02
)

In [0]:
silver_df.write.format("delta") \
    .mode("overwrite") \
        .option("overwriteSchema", "true") \
            .save("/Volumes/workspace/default/my_volume/delta/silver_Banking/bank_transactions")

In [0]:
gold_source_df = spark.read.format("delta") \
    .load("/Volumes/workspace/default/my_volume/delta/silver_Banking/bank_transactions")


In [0]:
from pyspark.sql.functions import sum 

gold_total_df = gold_source_df.agg(sum("amount").alias("total_amount"))

In [0]:
gold_risk_df = gold_source_df.groupBy("risk_categories").count()

In [0]:
gold_revenue_df = gold_source_df.groupBy("risk_categories") \
    .agg(sum("amount")).alias("total_amount")

In [0]:
from pyspark.sql.window import Window 
from pyspark.sql.functions import dense_rank,col

window_spec = Window.orderBy(
    col("amount").desc()
)

rank_df = silver_df.withColumn(
    "customer_rank",
    dense_rank().over(window_spec)
)

rank_df.show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+--------------+-------------+------+---------------+-------------+
|transaction_id|customer_name|amount|risk_categories|customer_rank|
+--------------+-------------+------+---------------+-------------+
|           103|         Mike|120000|      High Risk|            1|
|           105|          Tom| 70000|    Medium Risk|            2|
|           102|         Jane| 25000|       Low Risk|            3|
|           104|         Sara|  8000|       Low Risk|            4|
|           101|         John|  5000|       Low Risk|            5|
|           106|         Alex|     0|       Low Risk|            6|
+--------------+-------------+------+---------------+-------------+



In [0]:
rank_df.filter(
    col("customer_rank") == 1
).show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+--------------+-------------+------+---------------+-------------+
|transaction_id|customer_name|amount|risk_categories|customer_rank|
+--------------+-------------+------+---------------+-------------+
|           103|         Mike|120000|      High Risk|            1|
+--------------+-------------+------+---------------+-------------+



In [0]:
gold_df = rank_df
gold_df.write.format("delta") \
    .mode("overwrite") \
        .save("/Volumes/workspace/default/my_volume/delta/gold_Banking/bank_transactions")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
display(gold_df)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(
/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


transaction_id,customer_name,amount,risk_categories,customer_rank
103,Mike,120000,High Risk,1
105,Tom,70000,Medium Risk,2
102,Jane,25000,Low Risk,3
104,Sara,8000,Low Risk,4
101,John,5000,Low Risk,5
106,Alex,0,Low Risk,6


In [0]:
gold_source_df.printSchema()

root
 |-- transaction_id: long (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- amount: long (nullable = true)
 |-- risk_categories: string (nullable = true)



In [0]:
gold_source_df.show()

+--------------+-------------+------+---------------+
|transaction_id|customer_name|amount|risk_categories|
+--------------+-------------+------+---------------+
|           104|         Sara|  8000|       Low Risk|
|           102|         Jane| 25000|       Low Risk|
|           101|         John|  5000|       Low Risk|
|           105|          Tom| 70000|    Medium Risk|
|           103|         Mike|120000|      High Risk|
|           106|         Alex|     0|       Low Risk|
+--------------+-------------+------+---------------+



In [0]:
display(silver_df)

transaction_id,customer_name,amount,risk_categories
101,John,5000,Low Risk
102,Jane,25000,Low Risk
103,Mike,120000,High Risk
104,Sara,8000,Low Risk
105,Tom,70000,Medium Risk
106,Alex,0,Low Risk


In [0]:
display(bronze_df)

transaction_id,customer_name,amount
101,John,5000
102,Jane,25000
103,Mike,120000
104,Sara,8000
105,Tom,70000
103,Mike,120000
106,Alex,null


In [0]:
gold_source_df.printSchema()

root
 |-- transaction_id: long (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- amount: long (nullable = true)
 |-- risk_categories: string (nullable = true)



In [0]:
gold_risk_df.show()

+---------------+-----+
|risk_categories|count|
+---------------+-----+
|       Low Risk|    4|
|    Medium Risk|    1|
|      High Risk|    1|
+---------------+-----+



In [0]:
gold_revenue_df.show()

+---------------+-----------+
|risk_categories|sum(amount)|
+---------------+-----------+
|       Low Risk|      38000|
|    Medium Risk|      70000|
|      High Risk|     120000|
+---------------+-----------+



In [0]:
rank_df.show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+--------------+-------------+------+---------------+-------------+
|transaction_id|customer_name|amount|risk_categories|customer_rank|
+--------------+-------------+------+---------------+-------------+
|           103|         Mike|120000|      High Risk|            1|
|           105|          Tom| 70000|    Medium Risk|            2|
|           102|         Jane| 25000|       Low Risk|            3|
|           104|         Sara|  8000|       Low Risk|            4|
|           101|         John|  5000|       Low Risk|            5|
|           106|         Alex|     0|       Low Risk|            6|
+--------------+-------------+------+---------------+-------------+



Employee HR Analytics Pipeline

In [0]:
data = [
 (101,"John","IT",70000),
 (102,"Jane","HR",50000),
 (103,"Mike","IT",90000),
 (104,"Sara","Finance",120000),
 (105,"Tom","HR",60000),
 (106,"Alex","Finance",None),
 (103,"Mike","IT",90000)
]

columns = [
 "emp_id",
 "emp_name",
 "department",
 "salary"
]

bronze_df = spark.createDataFrame(
    data,
    columns
)

bronze_df.write.format("delta") \
    .mode("overwrite") \
        .save("/Volumes/workspace/default/my_volume/delta/bronze_HR/employees")
display(bronze_df)

emp_id,emp_name,department,salary
101,John,IT,70000
102,Jane,HR,50000
103,Mike,IT,90000
104,Sara,Finance,120000
105,Tom,HR,60000
106,Alex,Finance,null
103,Mike,IT,90000


In [0]:
silver_df = spark.read.format("delta") \
    .load("/Volumes/workspace/default/my_volume/delta/bronze_HR/employees")

In [0]:
silver_df = silver_df.dropDuplicates()

In [0]:
silver_df = silver_df.fillna({
    "salary":0
})

In [0]:
from pyspark.sql.functions import when,col 

silver_df = silver_df.withColumn(
    "salary_band",
    when(col("salary") >= 100000,"Executive")
    .when(col("salary") >= 70000,"Senior")
    .otherwise("Junior")
)

In [0]:
silver_df.write.format("delta") \
    .mode("overwrite") \
        .save("/Volumes/workspace/default/my_volume/delta/silver_HR/employees")

In [0]:
gold_source_df = spark.read.format("delta") \
    .load("/Volumes/workspace/default/my_volume/delta/silver_HR/employees")

In [0]:
from pyspark.sql.functions import avg

gold_avg_df = gold_source_df.groupBy("department") \
    .agg(avg("salary").alias("avg_salary"))


In [0]:
gold_count_df = gold_source_df.groupBy("department").count()

In [0]:
gold_band_df = gold_source_df.groupBy("salary_band").count()

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import dense_rank 

window_spec = Window.partitionBy("department") \
    .orderBy(col("salary").desc())

rank_df = gold_source_df.withColumn(
    "dept_rank",
    dense_rank().over(window_spec)
)

In [0]:
rank_df.filter(
    col("dept_rank") ==1
).show()

+------+--------+----------+------+-----------+---------+
|emp_id|emp_name|department|salary|salary_band|dept_rank|
+------+--------+----------+------+-----------+---------+
|   104|    Sara|   Finance|120000|  Executive|        1|
|   105|     Tom|        HR| 60000|     Junior|        1|
|   103|    Mike|        IT| 90000|     Senior|        1|
+------+--------+----------+------+-----------+---------+



In [0]:
gold_df = rank_df 

gold_df.write.format("delta") \
    .mode("overwrite") \
        .save("/Volumes/workspace/default/my_volume/delta/gold_HR/employees")

In [0]:
display(bronze_df)

emp_id,emp_name,department,salary
101,John,IT,70000
102,Jane,HR,50000
103,Mike,IT,90000
104,Sara,Finance,120000
105,Tom,HR,60000
106,Alex,Finance,null
103,Mike,IT,90000


In [0]:
display(silver_df)

emp_id,emp_name,department,salary,salary_band
101,John,IT,70000,Senior
102,Jane,HR,50000,Junior
103,Mike,IT,90000,Senior
104,Sara,Finance,120000,Executive
105,Tom,HR,60000,Junior
106,Alex,Finance,0,Junior


In [0]:
display(gold_df)

emp_id,emp_name,department,salary,salary_band,dept_rank
104,Sara,Finance,120000,Executive,1
106,Alex,Finance,0,Junior,2
105,Tom,HR,60000,Junior,1
102,Jane,HR,50000,Junior,2
103,Mike,IT,90000,Senior,1
101,John,IT,70000,Senior,2


In [0]:
display(gold_source_df)

emp_id,emp_name,department,salary,salary_band
102,Jane,HR,50000,Junior
106,Alex,Finance,0,Junior
104,Sara,Finance,120000,Executive
103,Mike,IT,90000,Senior
101,John,IT,70000,Senior
105,Tom,HR,60000,Junior


In [0]:
display(gold_avg_df)
display(gold_count_df)
display(gold_band_df)

department,avg_salary
HR,55000.0
Finance,60000.0
IT,80000.0


department,count
HR,2
Finance,2
IT,2


salary_band,count
Junior,3
Executive,1
Senior,2


Retail Inventory Analytics Pipeline

In [0]:
data = [
 (101,"Laptop",50,50000,"Delhi"),
 (102,"Phone",120,30000,"Mumbai"),
 (103,"Tablet",20,20000,"Delhi"),
 (104,"Monitor",10,15000,"Chennai"),
 (105,"Keyboard",200,2000,"Mumbai"),
 (103,"Tablet",20,20000,"Delhi"),
 (106,"Mouse",None,1000,"Delhi")
]

columns = [
 "product_id",
 "product_name",
 "stock_qty",
 "unit_price",
 "warehouse"
]
bronze_df = spark.createDataFrame(
    data,
    columns
)

bronze_df.write.format("delta") \
    .mode("overwrite") \
        .save("/Volumes/workspace/default/my_volume/delta/bronze_inventory/products")
display(bronze_df)

product_id,product_name,stock_qty,unit_price,warehouse
101,Laptop,50,50000,Delhi
102,Phone,120,30000,Mumbai
103,Tablet,20,20000,Delhi
104,Monitor,10,15000,Chennai
105,Keyboard,200,2000,Mumbai
103,Tablet,20,20000,Delhi
106,Mouse,null,1000,Delhi


In [0]:
silver_df = spark.read.format("delta") \
    .load("/Volumes/workspace/default/my_volume/delta/bronze_inventory/products")


In [0]:
silver_df = silver_df.dropDuplicates()

In [0]:
silver_df = silver_df.fillna({
    "stock_qty":0
})

In [0]:
from pyspark.sql.functions import when,col

silver_df = silver_df.withColumn(
    "stock_status",
    when(col("stock_qty") <= 20,"Low Stock")
    .otherwise("Sufficient Stock")
)

In [0]:
silver_df = silver_df.withColumn(
    "inverntory_value",
    col("stock_qty") * col("unit_price")
)

In [0]:
silver_df.write.format("delta") \
    .mode("overwrite") \
        .option("overwriteSchema", "true") \
            .save("/Volumes/workspace/default/my_volume/delta/silver_inventory/products")

In [0]:
display(silver_df)

product_id,product_name,stock_qty,unit_price,warehouse,stock_status,inverntory_value
101,Laptop,50,50000,Delhi,Sufficient Stock,2500000
102,Phone,120,30000,Mumbai,Sufficient Stock,3600000
103,Tablet,20,20000,Delhi,Low Stock,400000
104,Monitor,10,15000,Chennai,Low Stock,150000
105,Keyboard,200,2000,Mumbai,Sufficient Stock,400000
106,Mouse,0,1000,Delhi,Low Stock,0


In [0]:
gold_source_df = spark.read.format("delta") \
    .load("/Volumes/workspace/default/my_volume/delta/silver_inventory/products")   

In [0]:
from pyspark.sql.functions import sum

gold_inventory_df = gold_source_df.groupBy("warehouse") \
    .agg(sum("inverntory_value").alias("total_inventory_value"))


In [0]:
gold_stock_df = gold_source_df.groupBy("stock_status").count()

In [0]:
gold_warehouse_df = gold_source_df.groupBy("warehouse").count()

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import (
    row_number,
    rank,
    dense_rank
)

window_spec = Window.orderBy(col("inverntory_value").desc())

row_df = gold_source_df.withColumn(
    "row_num",
    row_number().over(window_spec)
)

rank_df = gold_source_df.withColumn(
    "rank_num",
    rank().over(window_spec)
)

dense_df = gold_source_df.withColumn(
    "dense_rank",
    dense_rank().over(window_spec)
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
dense_df.filter(
    col("dense_rank") == 1
).show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+----------+------------+---------+----------+---------+----------------+----------------+----------+
|product_id|product_name|stock_qty|unit_price|warehouse|    stock_status|inverntory_value|dense_rank|
+----------+------------+---------+----------+---------+----------------+----------------+----------+
|       102|       Phone|      120|     30000|   Mumbai|Sufficient Stock|         3600000|         1|
+----------+------------+---------+----------+---------+----------------+----------------+----------+



In [0]:
gold_df = dense_df 

gold_df.write.format("delta") \
    .mode("overwrite") \
        .option("overwriteSchema", "true") \
            .save("/Volumes/workspace/default/my_volume/delta/gold_inventory/products")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
display(gold_df)
display(gold_stock_df)
display(gold_warehouse_df)
display(gold_inventory_df)
display(row_df)
display(rank_df)
display(dense_df)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


product_id,product_name,stock_qty,unit_price,warehouse,stock_status,inverntory_value,dense_rank
102,Phone,120,30000,Mumbai,Sufficient Stock,3600000,1
101,Laptop,50,50000,Delhi,Sufficient Stock,2500000,2
103,Tablet,20,20000,Delhi,Low Stock,400000,3
105,Keyboard,200,2000,Mumbai,Sufficient Stock,400000,3
104,Monitor,10,15000,Chennai,Low Stock,150000,4
106,Mouse,0,1000,Delhi,Low Stock,0,5


stock_status,count
Sufficient Stock,3
Low Stock,3


warehouse,count
Mumbai,2
Delhi,3
Chennai,1


warehouse,total_inventory_value
Mumbai,4000000
Delhi,2900000
Chennai,150000


product_id,product_name,stock_qty,unit_price,warehouse,stock_status,inverntory_value,row_num
102,Phone,120,30000,Mumbai,Sufficient Stock,3600000,1
101,Laptop,50,50000,Delhi,Sufficient Stock,2500000,2
103,Tablet,20,20000,Delhi,Low Stock,400000,3
105,Keyboard,200,2000,Mumbai,Sufficient Stock,400000,4
104,Monitor,10,15000,Chennai,Low Stock,150000,5
106,Mouse,0,1000,Delhi,Low Stock,0,6


product_id,product_name,stock_qty,unit_price,warehouse,stock_status,inverntory_value,rank_num
102,Phone,120,30000,Mumbai,Sufficient Stock,3600000,1
101,Laptop,50,50000,Delhi,Sufficient Stock,2500000,2
103,Tablet,20,20000,Delhi,Low Stock,400000,3
105,Keyboard,200,2000,Mumbai,Sufficient Stock,400000,3
104,Monitor,10,15000,Chennai,Low Stock,150000,5
106,Mouse,0,1000,Delhi,Low Stock,0,6


product_id,product_name,stock_qty,unit_price,warehouse,stock_status,inverntory_value,dense_rank
102,Phone,120,30000,Mumbai,Sufficient Stock,3600000,1
101,Laptop,50,50000,Delhi,Sufficient Stock,2500000,2
103,Tablet,20,20000,Delhi,Low Stock,400000,3
105,Keyboard,200,2000,Mumbai,Sufficient Stock,400000,3
104,Monitor,10,15000,Chennai,Low Stock,150000,4
106,Mouse,0,1000,Delhi,Low Stock,0,5


Healthcare Analytics Pipeline

In [0]:
data = [
 (101,"John","Cardiology",50000),
 (102,"Jane","Neurology",120000),
 (103,"Mike","Orthopedics",30000),
 (104,"Sara","Cardiology",200000),
 (105,"Tom","Neurology",80000),
 (106,"Alex","Orthopedics",None),
 (102,"Jane","Neurology",120000)
]

columns = [
 "patient_id",
 "patient_name",
 "department",
 "treatment_cost"
]

bronze_df = spark.createDataFrame(
    data,
    columns
)

bronze_df.write.format("delta") \
    .mode("overwrite") \
        .save("/Volumes/workspace/default/my_volume/delta/bronze_healthcare/patients")

display(bronze_df)

patient_id,patient_name,department,treatment_cost
101,John,Cardiology,50000
102,Jane,Neurology,120000
103,Mike,Orthopedics,30000
104,Sara,Cardiology,200000
105,Tom,Neurology,80000
106,Alex,Orthopedics,null
102,Jane,Neurology,120000


In [0]:
silver_df = spark.read.format("delta") \
    .load("/Volumes/workspace/default/my_volume/delta/bronze_healthcare/patients")

In [0]:
silver_df = silver_df.dropDuplicates()

In [0]:
silver_df = silver_df.fillna({
    "treatment_cost": 0
})

In [0]:
from pyspark.sql.functions import when,col

silver_df = silver_df.withColumn(
    "risk_category",
    when(col("treatment_cost") >= 150000,"Critical")
    .when(col("treatment_cost") >= 80000,"High Risk")
    .otherwise("Normal")
)

In [0]:
silver_df.write.format("delta") \
    .mode("overwrite") \
        .save("/Volumes/workspace/default/my_volume/delta/silver_healthcare/patients")
display(silver_df)


patient_id,patient_name,department,treatment_cost,risk_category
101,John,Cardiology,50000,Normal
102,Jane,Neurology,120000,High Risk
103,Mike,Orthopedics,30000,Normal
104,Sara,Cardiology,200000,Critical
105,Tom,Neurology,80000,High Risk
106,Alex,Orthopedics,0,Normal


In [0]:
gold_source_df = spark.read.format("delta")\
    .load("/Volumes/workspace/default/my_volume/delta/silver_healthcare/patients")


In [0]:
from pyspark.sql.functions import avg

gold_avg_df = gold_source_df.groupBy("department") \
    .agg(avg("treatment_cost").alias("avg_treatment_cost"))

In [0]:
gold_count_df = gold_source_df.groupBy("department").count()

In [0]:
gold_risk_df = gold_source_df.groupBy("risk_category").count()

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import dense_rank,col

window_spec = Window.partitionBy("department") \
    .orderBy(col("treatment_cost").desc())

rank_df = gold_source_df.withColumn(
    "dept_rank",
    dense_rank().over(window_spec)
)


In [0]:
rank_df.filter(
    col("dept_rank")==1
).show()

+----------+------------+-----------+--------------+-------------+---------+
|patient_id|patient_name| department|treatment_cost|risk_category|dept_rank|
+----------+------------+-----------+--------------+-------------+---------+
|       104|        Sara| Cardiology|        200000|     Critical|        1|
|       102|        Jane|  Neurology|        120000|    High Risk|        1|
|       103|        Mike|Orthopedics|         30000|       Normal|        1|
+----------+------------+-----------+--------------+-------------+---------+



In [0]:
gold_df = rank_df

gold_df.write.format("delta") \
    .mode("overwrite") \
        .save("/Volumes/workspace/default/my_volume/delta/gold_healthcare/patients")

In [0]:
display(gold_df)

patient_id,patient_name,department,treatment_cost,risk_category,dept_rank
104,Sara,Cardiology,200000,Critical,1
101,John,Cardiology,50000,Normal,2
102,Jane,Neurology,120000,High Risk,1
105,Tom,Neurology,80000,High Risk,2
103,Mike,Orthopedics,30000,Normal,1
106,Alex,Orthopedics,0,Normal,2


In [0]:
display(gold_avg_df)
display(gold_count_df)
display(gold_risk_df)

department,avg_treatment_cost
Cardiology,125000.0
Orthopedics,15000.0
Neurology,100000.0


department,count
Cardiology,2
Orthopedics,2
Neurology,2


risk_category,count
Normal,3
Critical,1
High Risk,2


Telecom Customer Churn Analytics Pipeline

In [0]:
data = [
 (101,"John",24,800,0),
 (102,"Jane",12,600,1),
 (103,"Mike",36,1200,0),
 (104,"Sara",6,400,1),
 (105,"Tom",48,1500,0),
 (106,"Alex",18,None,1),
 (102,"Jane",12,600,1)
]

columns = [
 "customer_id",
 "customer_name",
 "tenure_months",
 "monthly_bill",
 "churn"
]

bronze_df = spark.createDataFrame(
    data,
    columns
)

bronze_df.write.format("delta") \
    .mode("overwrite") \
        .save("/Volumes/workspace/default/my_volume/delta/bronze_telecom/customers")
display(bronze_df)

customer_id,customer_name,tenure_months,monthly_bill,churn
101,John,24,800,0
102,Jane,12,600,1
103,Mike,36,1200,0
104,Sara,6,400,1
105,Tom,48,1500,0
106,Alex,18,null,1
102,Jane,12,600,1


In [0]:
silver_df = spark.read.format("delta") \
    .load("/Volumes/workspace/default/my_volume/delta/bronze_telecom/customers")


In [0]:
silver_df = silver_df.dropDuplicates()

In [0]:
silver_df = silver_df.fillna({
    "monthly_bill": 0
})


In [0]:
from pyspark.sql.functions import when,col

silver_df = silver_df.withColumn(
    "customer_segment",
    when(col("monthly_bill") >= 1000,"Premium")
    .when(col("monthly_bill") >=500,"Standard")
    .otherwise("Basic")
)

In [0]:
silver_df = silver_df.withColumn(
    "annual_revenue",
    col("monthly_bill") * 12
)

In [0]:
silver_df.write.format("delta") \
    .mode("overwrite") \
        .save("/Volumes/workspace/default/my_volume/delta/silver_telecom/customers")

In [0]:
gold_source_df = spark.read.format("delta") \
    .load("/Volumes/workspace/default/my_volume/delta/silver_telecom/customers")


In [0]:
gold_segment_df = gold_source_df.groupBy("customer_segment").count()

In [0]:
from pyspark.sql.functions import sum 

gold_revenue_df = gold_source_df.groupBy("customer_segment") \
    .agg(sum("annual_revenue").alias("total_annual_revenue"))


In [0]:
gold_churn_df = gold_source_df.groupBy("churn").count()


In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import dense_rank,col

window_spec = Window.partitionBy("customer_segment") \
    .orderBy(col("annual_revenue").desc())

rank_df = gold_source_df.withColumn(
    "Segment_rank",
    dense_rank().over(window_spec)
)

In [0]:
rank_df.filter(
    col("Segment_rank") ==1
).show()

+-----------+-------------+-------------+------------+-----+----------------+--------------+------------+
|customer_id|customer_name|tenure_months|monthly_bill|churn|customer_segment|annual_revenue|Segment_rank|
+-----------+-------------+-------------+------------+-----+----------------+--------------+------------+
|        104|         Sara|            6|         400|    1|           Basic|          4800|           1|
|        105|          Tom|           48|        1500|    0|         Premium|         18000|           1|
|        101|         John|           24|         800|    0|        Standard|          9600|           1|
+-----------+-------------+-------------+------------+-----+----------------+--------------+------------+



In [0]:
gold_df = rank_df 

gold_df.write.format("delta") \
    .mode("overwrite") \
        .save("/Volumes/workspace/default/my_volume/delta/gold_telecom/customers")

In [0]:
display(gold_df)

customer_id,customer_name,tenure_months,monthly_bill,churn,customer_segment,annual_revenue,Segment_rank
104,Sara,6,400,1,Basic,4800,1
106,Alex,18,0,1,Basic,0,2
105,Tom,48,1500,0,Premium,18000,1
103,Mike,36,1200,0,Premium,14400,2
101,John,24,800,0,Standard,9600,1
102,Jane,12,600,1,Standard,7200,2


In [0]:
display(gold_revenue_df)
display(gold_segment_df)
display(gold_churn_df)


customer_segment,total_annual_revenue
Basic,4800
Standard,16800
Premium,32400


customer_segment,count
Basic,2
Standard,2
Premium,2


churn,count
1,3
0,3
